# E1 · CLIP ViT-B/32 sobre la imagen completa (línea base)

**Pregunta.** ¿Un encoder visual preentrenado, sin ningún ajuste, ya ordena correctamente a la mascota buscada entre las de su especie?

| | |
|---|---|
| Pipeline | imagen completa → CLIP ViT-B/32 (OpenAI, revisión fija `3d74acf9`) → vector de 512 dimensiones, norma L2 |
| Entrenamiento | ninguno (*zero-shot*) |
| Modelo | `pettrace-embedder` versión 1 (`clip-vit-b32@v1`), en producción en la app |
| Dataset | `pettrace-reid@v3` (privado en Hugging Face; animales con códigos seudónimos, sin fotos en este notebook) |
| Protocolo | cada foto es una consulta; galería = las demás fotos de la misma especie (*leave-one-out*); cada animal puntúa con su mejor foto; Rank-1/5, mAP, EER |

Reproducir: `make image && make notebook NB=notebooks/e1_clip_zero_shot.ipynb` (ver el README). El notebook calcula todo paso a paso, registra una ejecución en MLflow con el commit del código, el commit del dataset y el sha256 del modelo, y adjunta figuras, tablas y este mismo notebook.

In [ ]:
import json, os

# Paths are relative to the repository root (nbconvert starts the kernel in notebooks/).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("TQDM_DISABLE", "1")
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from collections import Counter

import mlflow
import numpy as np
import onnxruntime as ort
import yaml
from IPython.display import Image, Markdown, display
from PIL import Image as PILImage, ImageOps

from reid import data, metrics
from reid.evaluation import evaluate, load_model, log_run
from reid.figures import explain, metrics_table, save

print("código  :", os.environ.get("GIT_SHA", "?")[:7], "(sin cambios locales)" if os.environ.get("GIT_DIRTY") == "false" else "(¡cambios sin commit!)")
print("MLflow  :", os.environ.get("MLFLOW_TRACKING_URI"))

EXPERIMENT, DATASET, MODEL, REPORT = "E1", "v3", "models:/pettrace-embedder/1", "reports/e1_v3"

## 1. Datos

In [ ]:
ds = data.load(DATASET)
bucket = lambda x: "oscura" if x < 0.25 else "tenue" if x < 0.4 else "normal"
rows = ["| Especie | Animales | Fotos | Fotos por animal | Luz normal / tenue / oscura | Particiones |", "|---|---|---|---|---|---|"]
for sp in ("cat", "dog"):
    ph = [p for p in ds.photos if p.species == sp]
    per = Counter(p.animal for p in ph)
    light = Counter(bucket(p.light) for p in ph)
    splits = Counter(ds.animals[a]["split"] for a in per)
    rows.append(f"| {sp} | {len(per)} | {len(ph)} | {min(per.values())}–{max(per.values())} | "
                f"{light['normal']} / {light['tenue']} / {light['oscura']} | {dict(splits)} |")
display(Markdown(f"**{ds.name}** · commit `{ds.commit}` · huella `{ds.fingerprint}`\n\n" + "\n".join(rows)))
groups = Counter(r["lookalike_group"] for r in ds.animals.values() if r["lookalike_group"])
display(Markdown("Grupos de parecido (negativos difíciles): " + ", ".join(
    f"`{g}`: " + ", ".join(a for a, r in sorted(ds.animals.items()) if r["lookalike_group"] == g) for g in sorted(groups))))

## 2. Modelo

Se descarga la versión registrada y se verifica que es el modelo que dice ser: el *golden set* (12 imágenes generadas) re-embebido con su propio pipeline debe coincidir con los embeddings guardados al registrarla. Luego se muestra cómo se construyó: su `pipeline.yaml` (el preprocesamiento exacto) y la ejecución que la registró (exportación de CLIP a ONNX desde la revisión fija de Hugging Face y paridad con PyTorch).

In [ ]:
model = load_model(MODEL)
display(Markdown("\n".join([
    "| | |", "|---|---|",
    f"| URI | `{model.uri}` (versión {model.registry_version} del registro) |",
    f"| Pipeline | `{model.model_version}` |",
    *[f"| sha256 `{f}` | `{h}` |" for f, h in model.sha256.items()],
    f"| Golden set | huella `{model.golden_fingerprint}`, {'✓ reproduce los embeddings guardados' if model.golden_ok else '✗ NO coincide'} |",
])))
assert model.golden_ok, "the downloaded model does not reproduce its golden fingerprint"

In [ ]:
# How the version was built: its own pipeline.yaml (the exact preprocessing) and its registration run.
spec = yaml.safe_load(open(os.path.join(model.dir, "pipeline.yaml")))
print(yaml.safe_dump(spec, sort_keys=False, allow_unicode=True))
client = mlflow.MlflowClient()
reg = client.get_run(client.get_model_version("pettrace-embedder", model.registry_version).run_id)
display(Markdown(
    f"Registrado por la ejecución `{reg.info.run_id[:8]}` (`{reg.info.run_name}`), script "
    f"`{os.path.basename(reg.data.tags.get('mlflow.source.name', ''))}` de este repositorio:\n\n"
    + "\n".join(["| Parámetro | Valor |", "|---|---|"] + [f"| {k} | `{v}` |" for k, v in sorted(reg.data.params.items())])
    + "\n\n" + "\n".join(["| Métrica del registro | Valor |", "|---|---|"]
                            + [f"| {k} | {v:.4g} |" for k, v in sorted(reg.data.metrics.items())])))
steps = {s["op"]: s for s in spec["steps"]}

## 3. Entrenamiento

No aplica: E1 es la línea base *zero-shot*. El encoder se usa tal como fue preentrenado; construirlo es exportarlo a ONNX (`scripts/register_v1.py`, ver la ejecución de registro arriba).

## 4. Evaluación, paso a paso

Todo el cálculo se hace aquí, a la vista: preprocesamiento, embeddings con ONNX Runtime, similitudes, ranking y métricas. Al final se comprueba que coincide exactamente con la librería `reid` que usan los scripts y las figuras.

### 4.1 Preprocesamiento

Los pasos y parámetros salen del `pipeline.yaml` de la versión; aquí están escritos explícitamente.

In [ ]:
pre = steps["clip_preprocess"]
RESAMPLE = {"bicubic": PILImage.Resampling.BICUBIC, "bilinear": PILImage.Resampling.BILINEAR}

def clip_preprocess(img):
    """pipeline.yaml → clip_preprocess, written out: shortest edge → 224, centre crop 224×224,
    scale to 0–1, normalise with CLIP's mean/std, HWC → CHW."""
    short = pre["resize_shortest_edge"]
    w, h = img.size
    size = (short, int(short * h / w)) if w <= h else (int(short * w / h), short)
    img = img.resize(size, RESAMPLE[pre["resample"]])
    ch, cw = pre["center_crop"]
    left, top = (img.width - cw) // 2, (img.height - ch) // 2
    img = img.crop((left, top, left + cw, top + ch))
    x = np.asarray(img, dtype=np.float32) * pre["rescale_factor"]
    x = (x - np.array(pre["mean"], dtype=np.float32)) / np.array(pre["std"], dtype=np.float32)
    return x.transpose(2, 0, 1)

def load(photo):
    return ImageOps.exif_transpose(PILImage.open(photo.path)).convert("RGB")

### 4.2 Embeddings

Cada foto preprocesada pasa por el encoder ONNX de la versión registrada (el mismo archivo, verificado por sha256, que sirve la app) y sale como un vector de 512 números.

In [ ]:
onnx_step = steps["onnx"]
encoder = ort.InferenceSession(os.path.join(model.dir, onnx_step["file"]), providers=["CPUExecutionProvider"])

def preprocess(img):
    return clip_preprocess(img)

p0 = ds.photos[0]
img0 = load(p0)
x0 = preprocess(img0)
print(f"ejemplo {os.path.basename(p0.path)} ({p0.animal}): {img0.size[0]}×{img0.size[1]} px → tensor {x0.shape}, "
      f"media {x0.mean():+.3f}, desviación {x0.std():.3f}")

chunks = []
for i in range(0, len(ds.photos), 32):
    X = np.stack([preprocess(load(p)) for p in ds.photos[i:i + 32]]).astype(np.float32)
    chunks.append(encoder.run([onnx_step["output"]], {onnx_step["input"]: X})[0])
E = np.concatenate(chunks)
norms = np.linalg.norm(E, axis=1)
print(f"embeddings: {E.shape[0]} fotos × {E.shape[1]} dimensiones; norma L2 entre {norms.min():.6f} y {norms.max():.6f} "
      "(el encoder ya normaliza, así que coseno = producto punto)")

### 4.3 Similitud

Como los vectores tienen norma 1, la similitud coseno entre dos fotos es su producto punto. Se calcula para todos los pares.

In [ ]:
animals = np.array([p.animal for p in ds.photos])
species = np.array([p.species for p in ds.photos])
S = E @ E.T  # similitud coseno entre todas las fotos

q = 0
same_sp = np.where((species == species[q]) & (np.arange(len(E)) != q))[0]
top = same_sp[np.argsort(-S[q, same_sp])][:5]
display(Markdown(f"Consulta de ejemplo: foto {q} de `{animals[q]}`. Las 5 fotos más parecidas de su especie:\n\n"
                 + "\n".join(["| Puesto | Animal | Similitud |", "|---|---|---|"]
                              + [f"| {i} | `{animals[g]}`{' ✓' if animals[g] == animals[q] else ''} | {S[q, g]:.3f} |"
                                 for i, g in enumerate(top, 1)])))

### 4.4 Ranking (*leave-one-out*)

Cada foto es una consulta, como un avistamiento. Se compara con todas las demás fotos **de su especie** (la propia sale de la galería); cada animal puntúa con su foto más parecida y se ordenan los animales. El *rank* es el puesto del animal correcto. Para el mAP se ordenan las fotos y se promedia la precisión en cada foto del mismo animal.

In [ ]:
# Leave-one-out within the species: the query photo leaves the gallery; each animal scores with its
# best photo (as the app does); the rank is the position of the true animal. AP ranks photos.
rows = []
for q in range(len(E)):
    gallery = np.where((species == species[q]) & (np.arange(len(E)) != q))[0]
    if len(set(animals[gallery])) < 2 or not (animals[gallery] == animals[q]).any():
        continue  # species with one animal, or an animal with a single photo: not evaluable
    best = {}
    for g in gallery:
        best[animals[g]] = max(best.get(animals[g], -np.inf), S[q, g])
    ranked = sorted(best, key=best.get, reverse=True)
    order = gallery[np.argsort(-S[q, gallery], kind="stable")]
    relevant = animals[order] == animals[q]
    precision_at_hits = np.cumsum(relevant)[relevant] / (np.flatnonzero(relevant) + 1)
    rows.append({"animal": animals[q], "species": species[q], "rank": ranked.index(animals[q]) + 1,
                 "ap": float(precision_at_hits.mean()), "ranked": ranked, "best": best})

r0 = rows[0]
display(Markdown(f"Ranking de animales para la consulta 0 (`{r0['animal']}`): "
                 + " > ".join(f"`{a}` {r0['best'][a]:.3f}" for a in r0["ranked"])
                 + f" → el animal correcto queda en el puesto **{r0['rank']}**; AP de la consulta {r0['ap']:.3f}."))

### 4.5 Métricas

**Rank-k** = proporción de consultas cuyo animal correcto está entre los k primeros. **mAP** = media de la precisión promedio por consulta. **EER** = tasa de error donde las falsas aceptaciones igualan a los falsos rechazos al variar un umbral fijo de similitud.

In [ ]:
ranks = np.array([r["rank"] for r in rows])
aps = np.array([r["ap"] for r in rows])
manual = {"global": {"queries": len(rows), "rank1": float(np.mean(ranks <= 1)), "rank5": float(np.mean(ranks <= 5)),
                     "mAP": float(aps.mean())}}
for sp in ("cat", "dog"):
    m = np.array([r["species"] == sp for r in rows])
    manual[sp] = {"queries": int(m.sum()), "rank1": float(np.mean(ranks[m] <= 1)), "rank5": float(np.mean(ranks[m] <= 5)),
                  "mAP": float(aps[m].mean())}

# EER: every pair of photos of the same species is "same animal" or "different animal"; a threshold
# t accepts a pair when its similarity ≥ t. FAR = different pairs accepted, FRR = same pairs
# rejected; the EER is where both are equal (the threshold is fitted on these same pairs: optimistic).
i, j = np.triu_indices(len(E), k=1)
keep = species[i] == species[j]
i, j = i[keep], j[keep]
same_pairs = S[i, j][animals[i] == animals[j]]
diff_pairs = S[i, j][animals[i] != animals[j]]
thresholds = np.unique(np.concatenate([same_pairs, diff_pairs]))
far = np.array([np.mean(diff_pairs >= t) for t in thresholds])
frr = np.array([np.mean(same_pairs < t) for t in thresholds])
k = int(np.argmin(np.abs(far - frr)))
manual_eer, manual_thr = (far[k] + frr[k]) / 2, thresholds[k]

display(Markdown("\n".join(["| | Consultas | Rank-1 | Rank-5 | mAP |", "|---|---|---|---|---|"]
    + [f"| {g} | {v['queries']} | {v['rank1']:.1%} | {v['rank5']:.1%} | {v['mAP']:.3f} |" for g, v in manual.items()])
    + f"\n\nPares: {len(same_pairs)} del mismo animal y {len(diff_pairs)} de animales distintos. "
    f"**EER {manual_eer:.1%}** con umbral {manual_thr:.3f}."))

### 4.6 Verificación

In [ ]:
# The same evaluation through the library (reid.evaluation / reid.metrics, used by the CLI and the
# figures below) must give exactly what was computed by hand above.
ev = evaluate(ds, model.pipeline, zero_shot=True)
lib = ev.result["test"]
assert np.allclose(E, ev.embeddings, atol=1e-6), "embeddings differ"
assert [r["rank"] for r in rows] == [r["rank"] for r in lib["per_query"]], "ranks differ"
for g in ("global", "cat", "dog"):
    for key in ("rank1", "rank5", "mAP"):
        assert abs(manual[g][key] - lib[g][key]) < 1e-9, (g, key)
assert abs(manual_eer - ev.result["val"]["eer"]) < 1e-9 and abs(manual_thr - ev.result["val"]["threshold"]) < 1e-9
print(f"✓ el cálculo manual coincide con la librería: embeddings (máx |Δ| {np.abs(E - ev.embeddings).max():.1e}), "
      f"{len(rows)} rankings, Rank-1/5, mAP y EER idénticos")

## 5. Resultados

In [ ]:
s1 = save(ev, model, REPORT)
display(Markdown(metrics_table({"E1": s1})))

In [ ]:
summ = json.load(open(os.path.join(REPORT, "summary.json")))
for name in ("confusion", "rank1_por_animal", "distribucion_scores", "cmc", "matriz_similitud"):
    display(Image(os.path.join(REPORT, f"{name}.png"), width=760))
    display(Markdown(explain(name, summ)))

## 6. Registro en MLflow

In [ ]:
run_id = log_run(ev, model, experiment=EXPERIMENT, artifacts=REPORT)
json.dump({"run_id": run_id, "experiment": EXPERIMENT, "dataset": ds.name, "dataset_commit": ds.commit,
           "model": MODEL, "git_sha": os.environ.get("GIT_SHA", "unknown")}, open(os.path.join(REPORT, "run.json"), "w"), indent=2)
display(Markdown(f"Ejecución MLflow: [`{run_id[:8]}`]({os.environ['MLFLOW_TRACKING_URI']}/#/experiments/"
                 f"{mlflow.get_experiment_by_name('pettrace-reid-eval').experiment_id}/runs/{run_id})"))

## 7. Conclusiones

- **El ranking funciona muy por encima del azar**: el animal correcto queda primero en ~73 % de las consultas frente a 14 % por azar, y casi siempre está entre los 5 primeros. Es la base de la decisión de producto: mostrar un *ranking* que el dueño confirma.
- **Un umbral fijo no separa identidades**: las similitudes de pares de animales distintos llegan a valores tan altos como las del mismo animal (EER ≈ 39 %). No se usan alertas automáticas por umbral.
- **Los errores se concentran en los grupos de parecido** (atigrados, blancos de pelo rizado), lo que motiva E2 (quitar el fondo) y E3–E5 (especializar con el dataset propio y con la descripción).
- Muestra pequeña (14 animales, 128 consultas): los resultados se leen como tendencia; los intervalos de confianza están pendientes (ver `TODO.md`).